# Rust 123: CLI and Processes

Chapter 23, and the program finally talks to *other programs*. `std::io::stdin` is the honest answer to Python's `input()`; `std::process::Command` is the honest answer to Python's `subprocess` — spawn a child, capture its **stdout** as data, read its exit code as a value. The second half of the chapter repeats the story in async form (`tokio::process`) and then builds real typed CLIs with `clap`. The running domain is the trading desk's own infrastructure: the chapter treats a `docker ps`-style tool as the thing you spawn, and — because this machine has no Docker — builds a **stub `docker` CLI inside the notebook** (§2) by spawning `rustc` itself, so every cell is deterministic, offline, and honest.

Everything here is a **runnable example**. Read the markdown, run the cell below it, and change nothing — the practice twin of this notebook is `23_cli-and-processes_p.ipynb`.

## Who this is written for

You are assumed to be a competent programmer arriving from Python and C++. `subprocess.run`, `input()` and `argparse` are old friends; C++ taught you `system()` and its total absence of structure. What is new in Rust: every process interaction is a **`Result` with a kind**, stdout is **raw bytes until you say otherwise**, a child is a **value you own** (`Child`), and the async twin shares the vocabulary but not the types. And the strictness has a reason: stringly-typed process plumbing is where shells hide their bugs — Rust makes the bytes, the codes and the streams explicit.

## How to read this notebook

- Markdown cells explain; code cells demonstrate. **Run each code cell as you reach it.** The desk directory `scratch23/` is created in §2 and removed in §9 — restart-and-run-from-the-top always works.
- Code that intentionally *fails* is shown as a comment, together with the diagnostic you would get. §12 is the catalogue.
- The running domain is the container fleet from the capstone: `md-cache`, `md-store`, `md-feed` — listed, inspected, and logged through the stub.
- One honesty note runs through the chapter: **the `docker` binary is a stub we compile ourselves**. It answers the same arguments with the same shapes as a real CLI — which is exactly what makes the `Command` mechanics real.

## The one idea to internalise

A subprocess is not a shell line — it is a **structured exchange between two programs**: argv in, bytes and an exit code out. Python's `subprocess` hides the plumbing behind keyword arguments; Rust hands you the pipes (`Stdio::piped()`), the raw bytes (`Vec<u8>`), and the wait state (`Child`) as separate values, so nothing is implicit. Once you see stdout as *data the child wrote to a pipe*, streaming logs, feeding filters, and building CLIs are all the same discipline.


### Contents

- [0. Notebook Conventions: The Machine Is the Network](#0-notebook-conventions-the-machine-is-the-network)
- [1. Reading stdin: `input()` and Its Limits](#1-reading-stdin-input-and-its-limits)
- [2. The Desk: Build a CLI with a CLI](#2-the-desk-build-a-cli-with-a-cli)
- [3. Running a Program: `.output()` and `Output`](#3-running-a-program-output-and-output)
- [4. Exit Codes Are Data](#4-exit-codes-are-data)
- [5. stdout and stderr as Data](#5-stdout-and-stderr-as-data)
- [6. Piping stdin Into a Child](#6-piping-stdin-into-a-child)
- [7. Environment and Working Directory](#7-environment-and-working-directory)
- [8. `spawn`, `Child`, and Streaming stdout](#8-spawn-child-and-streaming-stdout)
- [9. Building a Helper, Not Just Running One](#9-building-a-helper-not-just-running-one)
- [10. Async: `tokio::process`](#10-async-tokioprocess)
- [11. clap: Typed Command-Line Interfaces](#11-clap-typed-command-line-interfaces)
- [12. Reading the Compiler: E-Codes and Traps](#12-reading-the-compiler-e-codes-and-traps)
- [13. Mental Model: The Receipts](#13-mental-model-the-receipts)
- [14. Cheat Sheet](#14-cheat-sheet)
- [15. Review](#15-review)

If you are returning to this chapter later, §14 is the one-page summary, §12 the trap catalogue, and §13 the probe receipts behind every behavioural claim below.


## 0. Notebook Conventions: The Machine Is the Network

This notebook runs on the **evcxr** kernel, which wraps each cell into a generated program and compiles it incrementally. The standing rules from chapter 17 all hold: no `fn main`, state persists across cells, items are order-independent, one namespace per notebook, first cell that needs a name does the `use`. Four chapter-specific conventions:

- **Children are spawned by relative path.** The kernel's working directory is the notebook's directory, so `./scratch23/docker-stub.exe` resolves — a **bare name without `./` does not** (§8.3: Windows does not search the current directory, and that is a security feature).
- **The desk builds itself.** §2 writes the stub's Rust source into `scratch23/` and spawns `rustc` to compile it — the notebook uses `Command` to construct the very tool the rest of the chapter runs.
- **No `println!` inside `block_on`.** Under this kernel an async cell that prints from inside the runtime can wedge or tear down the session (§10, §12). Compute inside `block_on`, print after it. The probe receipts are in §13.
- **One runtime per async cell, scoped.** Create the runtime inside the block that uses it and let it drop before the cell returns. A second *top-level* runtime while an earlier one persists hangs the next async cell (§10, §12).
- **Capture results are owned.** evcxr persists your top-level `let`s between cells, and a binding holding a borrowed `Cow<str>` cannot persist — every text capture below ends in `.into_owned()` or lives inside a block (§5, §12).

### The crates

`clap` (feature `derive`) owns §11's typed CLI surface; `tokio` (feature `full` — already carrying the `process` module the async section uses) returns from chapters 16–19. Everything else is `std`. The stub is compiled with `rustc` directly, no cargo project needed.


In [ ]:
// 0.1 The chapter's crate cell: run once per kernel session, features included.
:dep clap = { version = "4", features = ["derive"] }
:dep tokio = { version = "1", features = ["full"] }

println!("crates loaded: clap, tokio");


In [ ]:
// 0.2 Smoke test: the kernel can reach the toolchain and spawn processes at all.
use std::process::Command;

let out = Command::new("rustc").arg("--version").output().unwrap();
println!("toolchain: {}", String::from_utf8_lossy(&out.stdout).trim());


### The kernel's stdin, settled by receipt

Before this chapter was written, a probe cell containing `stdin().read_line(&mut line)` was executed **headlessly through this exact kernel** (the same `evcxr_jupyter` these notebooks run on). It never returned: the probe timed out at 180 seconds with no bytes and no EOF. That is the honest answer to "what does Python's `input()` map to here":

| Fact | Receipt |
|---|---|
| The kernel host owns stdin | A notebook cell **blocks forever** on `read_line` — there is no console to type into, and no pipe EOF to release it |
| `stdin()` compiles fine | The call is legal; the *wait* is the trap |
| Where it does work | In a **binary you build and run** — and in any child whose stdin *you* pipe (§1.2 gives the receipt) |

So §1 teaches the `input()` shape as the pattern every shipped binary uses, and demonstrates the mechanics where they actually run — in children. The practice notebook keeps its check blocks free of any kernel-stdin read for the same reason.


## 1. Reading stdin: `input()` and Its Limits

Python's `input(prompt)` hides three steps: write the prompt to stdout, read a **line** from stdin, strip the trailing newline. Rust does not hide them — `std::io::stdin()` is a handle, `read_line` fills a `String` you own, and the return value is the byte count: `Ok(0)` means EOF, the `Err` arm is an I/O failure. Parsing and validation stay explicit, which is exactly where Python programs grow their own `while True: try: ...` loops.

### The typed pattern every binary uses


In [ ]:
// 1.1 The input() pattern, typed: read a line, detect EOF, parse -- nothing hidden.
use std::io::BufRead;

/// Read one line from `input`, parse it as f64. The shape of Python's
/// `float(input(prompt))` with the hiding removed: EOF is `Ok(0)`, a bad
/// number is a data error carrying the offending text. The reader is a
/// parameter because the kernel's own stdin is dead (§0) -- and because
/// a function that takes `dyn BufRead` is testable against any source.
fn read_number(input: &mut dyn BufRead) -> Result<f64, String> {
    let mut line = String::new();
    let n = input.read_line(&mut line).map_err(|e| e.to_string())?;
    if n == 0 {
        return Err("end of input (EOF)".to_string());
    }
    line.trim().parse::<f64>()
        .map_err(|e| format!("'{}' is not a number ({e})", line.trim()))
}

let mut demo_input = "4125.5\nnot-a-number\n".as_bytes();
println!("{:?}", read_number(&mut demo_input));
println!("{:?}", read_number(&mut demo_input));


Two errors, two shapes: `n == 0` is **EOF** (the stream ended — Ctrl-Z/Ctrl-D at a console), and a failed `parse` is a **data problem** with the offending text in the message. The `dyn BufRead` parameter is the same trick chapter 06's iterators taught: program against the trait, and the function accepts a file, a socket, or — as here — a `&[u8]` slice that stands in for typed input.

### The same mechanics, running for real in a child

A child process has real pipes. The receipt below writes text **into a child's stdin** and reads what it answers — the plumbing `input()` stands on, exercised end to end. §6 teaches this API properly; the cell here only proves bytes flow.


In [ ]:
// 1.2 The stdin receipt: bytes we write reach a child; Stdio and Write arrive here.
use std::io::Write;
use std::process::Stdio;

let mut child = Command::new("cmd").args(["/C", "findstr", "typed"])
    .stdin(Stdio::piped()).stdout(Stdio::piped()).spawn().unwrap();
{
    let mut pipe = child.stdin.take().unwrap();
    pipe.write_all(b"typed line reaches a child\n").unwrap();
} // dropping the handle closes the child's stdin: EOF
let out = child.wait_with_output().unwrap();
println!("child read our stdin: {:?}", String::from_utf8_lossy(&out.stdout));


## 2. The Desk: Build a CLI with a CLI

Chapters 17–19 built their servers from std primitives *inside* the notebook. This chapter's counterpart is a **command-line program**: the stub `docker` CLI, compiled from source the notebook writes. Building it with `Command::new("rustc")` is not a stunt — it is the same move every build system makes, and it exercises the whole §3 API before the teaching examples begin.

### The stub's contract

`docker-stub ps` lists three containers (the capstone fleet); `ps --format "{{json .}}"` prints one **JSON object per line** — the shape real `docker ps --format` uses, which §5 and the practice notebook consume; `version` prints a version line (overridable via `STUB_VERSION`, §7); `logs <id>` writes three lines to stdout **and one to stderr** (§5 needs both); `sleep <secs>` runs long enough to be killed (§8); anything else exits **2** with usage on stderr.


In [ ]:
// 2.1 Write the stub source into scratch23/, compile it with rustc, time the build.
use std::fs;
use std::time::Instant;

// One balanced line per source line: joined and written as the stub's .rs file.
let stub_lines = [
    "struct Container { id: &'static str, image: &'static str, names: &'static str, status: &'static str, ports: &'static str }",
    "const CONTAINERS: [Container; 3] = [",
    "    Container { id: \"3f1a9c2e7b01\", image: \"redis:7-alpine\", names: \"md-cache\", status: \"Up 2 hours\", ports: \"0.0.0.0:6379->6379/tcp\" },",
    "    Container { id: \"8d44b0aa19ce\", image: \"postgres:16\", names: \"md-store\", status: \"Up 2 hours\", ports: \"0.0.0.0:5432->5432/tcp\" },",
    "    Container { id: \"c07e55d3f8a2\", image: \"ghcr.io/acme/feed:3\", names: \"md-feed\", status: \"Exited (137) 5 minutes ago\", ports: \"\" },",
    "];",
    "fn main() { let args: Vec<String> = std::env::args().skip(1).collect();",
    "    match args.first().map(|s| s.as_str()).unwrap_or(\"\") {",
    "        \"ps\" => ps(&args),",
    "        \"version\" => println!(\"Docker version {}, build f4e3f4e\",",
    "            std::env::var(\"STUB_VERSION\").unwrap_or_else(|_| \"23.0.0-stub\".into())),",
    "        \"logs\" => logs(args.get(1).map(|s| s.as_str()).unwrap_or(\"\")),",
    "        \"sleep\" => sleep(args.get(1).and_then(|s| s.parse().ok()).unwrap_or(1.0)),",
    "        other => { eprintln!(\"docker: 'docker {}' is not a docker-stub command.\", other);",
    "            eprintln!(\"usage: docker-stub <ps|version|logs|sleep> [args]\");",
    "            std::process::exit(2); }",
    "    } }",
    "fn logs(id: &str) {",
    "    println!(\"[{id}] feed: connected to upstream\");",
    "    println!(\"[{id}] feed: 5120 ticks replayed\");",
    "    eprintln!(\"[{id}] feed: upstream disconnect, reconnecting...\");",
    "    println!(\"[{id}] feed: resumed at seq 5121\"); }",
    "fn sleep(secs: f64) { let mut done = 0.0;",
    "    while done < secs { std::thread::sleep(std::time::Duration::from_millis(100)); done += 0.1; }",
    "    println!(\"slept {secs}s\"); }",
    "fn ps(args: &[String]) {",
    "    let mut json_mode = false;",
    "    for a in args { if a == \"--format\" { json_mode = true; } }",
    "    if json_mode { for c in &CONTAINERS {",
    "        println!(\"{{\\\"ID\\\":\\\"{}\\\",\\\"Image\\\":\\\"{}\\\",\\\"Names\\\":\\\"{}\\\",\\\"State\\\":\\\"{}\\\",\\\"Status\\\":\\\"{}\\\",\\\"Ports\\\":\\\"{}\\\"}}\",",
    "            c.id, c.image, c.names,",
    "            if c.status.starts_with(\"Up\") { \"running\" } else { \"exited\" }, c.status, c.ports); }",
    "    } else {",
    "        println!(\"{:<12} {:<26} {:<10} {}\", \"CONTAINER ID\", \"IMAGE\", \"NAMES\", \"STATUS\");",
    "        for c in &CONTAINERS { println!(\"{:<12} {:<26} {:<10} {}\", c.id, c.image, c.names, c.status); } } }",
];
fs::create_dir_all("scratch23").unwrap();
fs::write("scratch23/docker-stub.rs", stub_lines.join("\n")).unwrap();

let t0 = Instant::now();
let build = Command::new("rustc")
    .args(["--edition", "2024", "-O", "-o", "scratch23/docker-stub.exe", "scratch23/docker-stub.rs"])
    .output().unwrap();
assert!(build.status.success(), "stub build failed: {}", String::from_utf8_lossy(&build.stderr));
println!("stub built in {:?}", t0.elapsed());


`rustc --edition 2024 -O` compiles the single file straight to an exe — no cargo project, no dependencies. On this toolchain the build lands well under a second. The source lines ride in the notebook as a `Vec` of one-line strings, joined and written in one `fs::write` — chapter 07's write API feeding chapter 23's spawn API.

### The tour

Five invocations, one loop. Note the two channels: `logs` narrates on stderr while its data lands on stdout, and `bogus` shows a usage error where a shell would show... a usage error.


In [ ]:
// 2.2 The demo tour: five subcommands through the same Command shape.
for argv in [
    vec!["version"],
    vec!["ps"],
    vec!["ps", "--format", "{{json .}}"],
    vec!["logs", "3f1a9c2e7b01"],
    vec!["bogus"],
] {
    let out = Command::new("./scratch23/docker-stub.exe").args(&argv).output().unwrap();
    println!("--- docker {:?} -> code {} ---", argv, out.status.code().unwrap_or(-1));
    print!("{}", String::from_utf8_lossy(&out.stdout));
    eprint!("{}", String::from_utf8_lossy(&out.stderr));
}


## 3. Running a Program: `.output()` and `Output`

`Command::new(program)` builds a command **description**; nothing has run yet — the builder methods (`.args`, `.env`, `.current_dir`, `.stdin`) all take `self` by value and hand it back, so the chain is one expression. `.output()` runs it to completion and returns `io::Result<Output>`, where `Output` bundles everything a finished child left behind:

| Field | Type | Meaning |
|---|---|---|
| `status` | `ExitStatus` | the exit code, platform-wrapped |
| `stdout` | `Vec<u8>` | what the child wrote to stdout |
| `stderr` | `Vec<u8>` | what the child wrote to stderr |

That `Result` answers **"did we launch anything at all?"** — it is `Err` before a single byte exists.


In [ ]:
// 3.1 .output(): run to completion, then inspect the three fields.
let out = Command::new("./scratch23/docker-stub.exe").arg("version").output().unwrap();
println!("status  = {:?}", out.status);
println!("stdout  = {:?}", String::from_utf8_lossy(&out.stdout));
println!("stderr  = {:?}", String::from_utf8_lossy(&out.stderr));
println!("bytes   = {} stdout bytes", out.stdout.len());


The `Err` arm arrives when the **program itself** cannot be launched: not found, not executable, permission denied. This is the `subprocess` difference that bites: Python raises `FileNotFoundError` at `.run()`; Rust returns `Err` with `ErrorKind::NotFound` — and because it is a value, you handle it like any other I/O miss from chapter 07.

### The launch failure, and what unwrap does to it


In [ ]:
// 3.2 A missing program is Err(ErrorKind::NotFound) -- a value, not a crash.
let missing = Command::new("definitely-missing-xyz-9q").output();
match missing {
    Ok(_) => println!("unexpectedly launched"),
    Err(e) => println!("launch failed: kind={:?} message='{e}'", e.kind()),
}
// The cheap-and-dirty alternative panics with the error's Debug form:
//   called `Result::unwrap()` on an `Err` value: Error { kind: NotFound, message: "program not found" }
let _ = Command::new("definitely-missing-xyz-9q").output();


## 4. Exit Codes Are Data

`ExitStatus` is deliberately opaque: its `code()` returns `Option<i32>`, because some termination paths have **no code at all** (a signal-killed child on Unix). The two workhorses: `.success()` for the yes/no question, `.code()` for the specific number. On Windows the mapping is direct — the OS exit code *is* the number.

### Reading the trichotomy

The stub speaks the CLI convention: **0** success, **2** usage error. Real CLIs add **1** for runtime failure; the exit code is the only channel that survives redirection, so scripts build on it.


In [ ]:
// 4.1 success() and code() over the stub's exit-code range.
for argv in [vec!["version"], vec!["ps"], vec!["bogus"]] {
    let out = Command::new("./scratch23/docker-stub.exe").args(&argv).output().unwrap();
    println!("docker {:?}: code={:?} success={}", argv, out.status.code(), out.status.success());
}


### `output()` vs `status()` vs `spawn()`

Three ways to run to completion (or not), three opinions about the streams:

| Method | Streams | Returns | Use when |
|---|---|---|---|
| `.output()` | both **captured** | `Output` (bytes) | you want the child's data |
| `.status()` | both **inherited** | `ExitStatus` | you only need the code |
| `.spawn()` | your choice via `Stdio` | `Child` | you want control: stream, pipe, kill |

There is no `.run()` — Python's `subprocess.run` maps to `.output()` (or `.status()` when the output is noise). The inherited case is the one to be careful with in a notebook: the child writes **straight into the cell's output**, which is exactly what §4.2 demonstrates — its version line appears above the printed result, unasked.


In [ ]:
// 4.2 status(): streams inherited (the version line appears above), code returned.
let st = Command::new("./scratch23/docker-stub.exe").arg("version").status().unwrap();
println!("status-only run: code={:?} success={}", st.code(), st.success());


## 5. stdout and stderr as Data

`Output.stdout` is `Vec<u8>`: the child's bytes, **not** a `String`. Text is a decoding step, and `String::from_utf8_lossy` is the forgiving decoder — valid UTF-8 passes through untouched, bad bytes become `U+FFFD` instead of an error. The honest name for its return type is `Cow<str>`: borrowed when the bytes were already valid UTF-8, owned when repair was needed.

### The kernel rule this chapter runs under

A **cell-level binding must be owned**. evcxr persists your top-level `let`s between cells, and a borrowed `Cow<str>` has no `'static` lifetime to persist — the cell is refused with *"The variable `text` contains a reference with a non-static lifetime so can't be persisted."* The fix is one method call, and it is why every text-capturing cell in this chapter (and the practice notebook) ends in `.into_owned()` on its way out of a capture.


In [ ]:
// 5.1 Separate channels: data on stdout, narration on stderr -- visible in the types.
let out = Command::new("./scratch23/docker-stub.exe").args(["logs", "3f1a9c2e7b01"]).output().unwrap();
let stdout_text = String::from_utf8_lossy(&out.stdout).into_owned();
let stderr_text = String::from_utf8_lossy(&out.stderr).into_owned();
for line in stdout_text.lines() { println!("  out| {line}"); }
for line in stderr_text.lines() { println!("  err| {line}"); }
println!("counts: {} stdout, {} stderr", stdout_text.lines().count(), stderr_text.lines().count());


Line endings are the **child's** business: a Rust `println!` writes bare `
` (the receipt below), while `cmd.exe` and many native tools write `
` into pipes — §13 keeps both receipts. Either way `str::lines()` strips the `'
'` and any preceding `'
'`, which is why §2.2's JSON lines came out clean. The two trimming tools and their difference:

| Tool | Eats | Keeps |
|---|---|---|
| `.lines()` | `\n` **and** a preceding `\r` | the line text |
| `.trim()` | all leading/trailing whitespace | the middle |

### From bytes to lines, the standard chain


In [ ]:
// 5.2 The capture chain: bytes -> lossy String -> lines; endings handled by lines().
// (the block keeps the borrowed intermediates out of the kernel's persistence check)
let (n_lines, first_clean, ends_lf) = {
    let out = Command::new("./scratch23/docker-stub.exe")
        .args(["ps", "--format", "{{json .}}"]).output().unwrap();
    let text = String::from_utf8_lossy(&out.stdout).into_owned();
    let lines: Vec<&str> = text.lines().collect();
    (lines.len(), lines[0].ends_with('}'), out.stdout.ends_with(b"\r\n"))
};
println!("{n_lines} JSON lines; first ends clean: {first_clean}; raw ends LF: {ends_lf}");


## 6. Piping stdin Into a Child

By default a child **inherits** your stdin. `Stdio::piped()` gives the parent the other end of the pipe as a `ChildStdin` you can write to — and the write side must **close** before a line-oriented reader sees EOF. Rust's idiom: `.take()` the handle, write inside a block, let the block end drop it.

### The classic filter: feed `findstr`, read its stdout

`findstr` is Windows' line filter (Unix: `grep`). We hand it three price lines, close the pipe, and collect its output — a full producer→filter→consumer pipeline with no shell. (`Write` and `Stdio` persist from §1.2.)


In [ ]:
// 6.1 Piped stdin: write, drop to signal EOF, wait_with_output to collect.
let mut child = Command::new("cmd").args(["/C", "findstr", "AAPL"])
    .stdin(Stdio::piped()).stdout(Stdio::piped()).spawn().unwrap();
{
    let mut pipe = child.stdin.take().unwrap();
    pipe.write_all(b"MSFT 231.75\nAAPL 4125.50\nNVDA 187.20\n").unwrap();
} // <- the drop that matters: closing stdin is the child's EOF
let out = child.wait_with_output().unwrap();
println!("filtered: {:?}", String::from_utf8_lossy(&out.stdout));


Forget the inner block — hold the `ChildStdin` while calling `wait_with_output` — and many children deadlock: they wait for EOF you only send by dropping, you wait for their exit. Python's `subprocess` grows the same warning in its docs; Rust makes the drop the *visible* mechanism instead of a footnote.

### The three `Stdio` settings

| Setting | Child's stream | Parent's handle |
|---|---|---|
| `inherit` (default) | the kernel's/console's | none |
| `piped()` | a new pipe | `child.stdin`/`.stdout`/`.stderr` (after `.take()`) |
| `null()` | the void | none, reads return EOF |

`.wait_with_output()` is the convenience that pairs with `piped()`: it drains the pipe to the end **and** returns the same `Output` as §3 — it is `.output()` for a child you had to touch mid-flight.


## 7. Environment and Working Directory

A child inherits a **copy** of your environment; `.env(k, v)` adds or overrides one key for this child only, `.envs(iter)` a batch, `.env_remove(k)` deletes one. Chapter 07's `env::set_var` was `unsafe` because *your own* environment is shared with other threads — the child's copy has no such hazard, so `.env` is safe.

### `STUB_VERSION`: one variable, one child

The stub reads `STUB_VERSION` and falls back to a default — the same `unwrap_or_else` pattern as chapter 07's `var()` recipe, seen this time from the parent's side.


In [ ]:
// 7.1 .env() reaches exactly one child; the parent's environment is untouched.
let plain = Command::new("./scratch23/docker-stub.exe").arg("version").output().unwrap();
let patched = Command::new("./scratch23/docker-stub.exe").arg("version")
    .env("STUB_VERSION", "99.9.9-test").output().unwrap();
println!("default: {}", String::from_utf8_lossy(&plain.stdout).trim());
println!("patched: {}", String::from_utf8_lossy(&patched.stdout).trim());


`.current_dir(dir)` sets where the child **runs** — its `cwd`, resolved for relative file access, inherited by its own children. It does **not** change where the exe is looked up (§8.3 settles lookup separately).

### The security aside: `.env_clear()`

CLIs that shell out have historically leaked secrets through inherited environments (`AWS_*`, tokens, proxy credentials). `.env_clear()` starts the child from nothing; `.envs(...)` then re-adds only what the child needs. One line of defence, built into the builder.


In [ ]:
// 7.2 current_dir: the child's pwd moves; path lookup is a separate question (§8.3).
let out = Command::new("cmd").args(["/C", "cd"])
    .current_dir("C:/Windows/Temp").output().unwrap();
println!("child cwd: {}", String::from_utf8_lossy(&out.stdout).trim());
let here = Command::new("cmd").args(["/C", "cd"]).output().unwrap();
println!("our cwd:   {}", String::from_utf8_lossy(&here.stdout).trim());


## 8. `spawn`, `Child`, and Streaming stdout

`.output()` is all-or-nothing: nothing until exit, everything at once. Long-running children need the other shape — `.spawn()` returns a `Child` **immediately**, and its streams become parents you own: `child.stdout` is an `Option<ChildStdout>`, `take()`n once. Wrap it in `BufReader` and it is chapter 07's line API again — this time the lines come from another process. (`BufRead` persists from §1.1.)

### Streaming a log while the child still runs


In [ ]:
// 8.1 spawn + take + BufReader::lines: stream the child's stdout as it writes.
use std::io::BufReader;

let mut child = Command::new("./scratch23/docker-stub.exe")
    .args(["logs", "3f1a9c2e7b01"])
    .stdout(Stdio::piped()).spawn().unwrap();
let stdout = child.stdout.take().unwrap();
let mut n = 0;
for line in BufReader::new(stdout).lines() {
    println!("line {n}: {}", line.unwrap());
    n += 1;
}
let st = child.wait().unwrap();
println!("streamed {n} lines, exit {:?}", st.code());


### `try_wait`, `wait`, `kill` — the lifecycle

| Call | Blocks? | Returns |
|---|---|---|
| `try_wait()` | never | `Ok(Some(status))` if exited, `Ok(None)` if still running |
| `wait()` | until exit | the final `ExitStatus` |
| `kill()` | never | `Ok(())` — the request was sent; `wait()` still needed |

Killed children still need `wait()`, or they linger until the `Child` value drops. Two platform receipts from this toolchain: on **Windows a killed child reports `code = Some(1)`** — the OS maps termination to an exit code, where Unix reports `None` (signalled); and **`kill()` after natural exit is `Ok(())`** — the request succeeds idempotently.


In [ ]:
// 8.2 The lifecycle: probe, kill, reap -- with the Windows code-1 receipt.
use std::time::Duration;

let mut child = Command::new("./scratch23/docker-stub.exe").args(["sleep", "30"]).spawn().unwrap();
std::thread::sleep(Duration::from_millis(300));
println!("still running: {}", child.try_wait().unwrap().is_none());
child.kill().unwrap();
let st = child.wait().unwrap();
println!("killed on Windows: code={:?} success={}", st.code(), st.success());

// kill-after-exit is Ok(()): the request is idempotent.
let mut done = Command::new("./scratch23/docker-stub.exe").arg("version").spawn().unwrap();
let _ = done.wait().unwrap();
println!("kill after exit: {:?}", done.kill());


### How the program name is looked up

`Command::new` resolves the program like a shell would — with one sharp edge:

| Argument | Resolved how |
|---|---|
| contains `/` or `\` (or starts `./`) | **as a path**, relative to `current_dir` |
| bare name (`rustc`, `cmd`) | searched in `PATH` **only** |

A bare name does **not** fall back to the current directory — on Windows explicitly so, on Unix because `PATH` never includes `.` in sane setups. The security story writes itself: a script run from an attacker-writable directory must not launch bait just because a matching name sits there. The receipt, run where the stub lives:


In [ ]:
// 8.3 Lookup receipts: ./ resolves, the bare name does not.
let rel = Command::new("./scratch23/docker-stub.exe").arg("version").output().unwrap();
println!("'./scratch23/docker-stub.exe' -> ok: {}", String::from_utf8_lossy(&rel.stdout).trim());
let bare = Command::new("docker-stub.exe").arg("version").output();
match bare {
    Ok(_) => println!("bare name launched (unexpected here)"),
    Err(e) => println!("bare 'docker-stub.exe' -> Err kind={:?} ({e})", e.kind()),
}


## 9. Building a Helper, Not Just Running One

One receipt worth keeping, and it is the *opposite* of the cargo intuition: a raw `rustc` single-file build has **no incremental cache** — the second build costs the same as the first. It is cargo's `target/` bookkeeping that makes rebuilds cheap, not the compiler.

### Rebuild and compare


In [ ]:
// 9.1 The compose chain, again: rustc on the persisted source, then run it.
let t0 = std::time::Instant::now();
let build = Command::new("rustc")
    .args(["--edition", "2024", "-O", "-o", "scratch23/docker-stub.exe", "scratch23/docker-stub.rs"])
    .output().unwrap();
assert!(build.status.success(), "stderr: {}", String::from_utf8_lossy(&build.stderr));
let build_ms = t0.elapsed().as_millis();

let run = Command::new("./scratch23/docker-stub.exe").arg("version").output().unwrap();
println!("rebuild in {build_ms}ms -> {}", String::from_utf8_lossy(&run.stdout).trim());


## 10. Async: `tokio::process`

Chapter 16's rule returns: async earns its keep when you **wait on many things at once**. A fleet of children is exactly that. `tokio::process::Command` mirrors the sync API method-for-method — the differences are `await` on every blocking call, and the runtime must be yours (`Runtime::new()` + `block_on`, the kernel-friendly shape from chapter 16) — **scoped**: created inside the block that uses it, dropped before the cell returns (`let out = { let rt = Runtime::new().unwrap(); rt.block_on(async { .. }) };`). A second top-level runtime while an earlier one persists wedged kernel probes repeatedly (§12); scoped, the same cells pass.

### The one chapter rule: print *after* `block_on`

Under this kernel, a `println!` **inside** the runtime can wedge the cell or tear down the session — evcxr captures output on the main thread, and the runtime's workers race it. The receipts live in §13; the shape here computes inside `block_on`, returns owned data, prints outside. (Chapters 16–22 followed the same shape; here it is not style but survival.)

### `.output().await`: the async twin of §3


In [ ]:
// 10.1 tokio::process::Command::output().await -- same Output, awaited.
// (Stdio persists from §1.2 -- the same std type tokio's builder consumes.)
let text: String = {
    let rt = tokio::runtime::Runtime::new().unwrap();
    rt.block_on(async {
        let out = tokio::process::Command::new("./scratch23/docker-stub.exe")
            .args(["ps", "--format", "{{json .}}"])
            .stdout(Stdio::piped())
            .output()
            .await
            .unwrap();
        String::from_utf8_lossy(&out.stdout).into_owned()
    })
};
println!("async ps: {} JSON lines", text.lines().count());
println!("first: {}", text.lines().next().unwrap_or(""));


### Streaming, async edition

The `Child::stdout` dance is the same; the reader is tokio's, and `AsyncBufReadExt::next_line()` replaces the `for` loop — each line is an `await`, so other tasks could interleave while this one waits. Only stdout is piped — pipe what you read.


In [ ]:
// 10.2 Async line streaming: BufReader + next_line over the child's stdout.
// The reader type's NAME is taken -- §8.1 imported std's BufReader, and the
// one-namespace rule allows one claimant. The trait imports; the struct is spelled.
// stdout piped only; stderr inherited -- pipe what you read. The runtime
// below is scoped (it drops before the cell returns): a second top-level
// Runtime while an earlier one persists wedges this kernel (§12).
use tokio::io::AsyncBufReadExt;

let (n, ok) = {
    let rt = tokio::runtime::Runtime::new().unwrap();
    rt.block_on(async {
        let mut child = tokio::process::Command::new("./scratch23/docker-stub.exe")
            .args(["logs", "3f1a9c2e7b01"])
            .stdout(Stdio::piped())
            .spawn()
            .unwrap();
        let stdout = child.stdout.take().unwrap();
        let mut lines = tokio::io::BufReader::new(stdout).lines();
        let mut n = 0;
        while let Some(_line) = lines.next_line().await.unwrap() {
            n += 1;
        }
        let st = child.wait().await.unwrap();
        (n, st.success())
    })
};
println!("async stream: {n} lines, ok={ok}");


### Timeouts and `kill_on_drop`

Chapter 16's `timeout` wraps any future — including `child.wait()`. On miss, the future is **dropped mid-wait**, and the sync chapter's rule (§8.2: still must reap) bites differently: `kill_on_drop(true)` makes the drop itself kill the child, so an abandoned `Child` cannot linger. The pair is the async answer to "don't wait forever, don't leak either".


In [ ]:
// 10.3 timeout() around wait(); kill_on_drop cleans up on the miss path.
let outcome: String = {
    let rt = tokio::runtime::Runtime::new().unwrap();
    rt.block_on(async {
        let mut child = tokio::process::Command::new("./scratch23/docker-stub.exe")
            .args(["sleep", "30"])
            .stdout(Stdio::piped())
            .kill_on_drop(true)
            .spawn()
            .unwrap();
        match tokio::time::timeout(std::time::Duration::from_millis(400), child.wait()).await {
            Ok(res) => format!("completed: success={}", res.unwrap().success()),
            Err(_elapsed) => {
                let _ = child.kill().await; // belt: kill_on_drop would do this on drop
                "timed out (400ms) and killed".to_string()
            }
        }
    })
};
println!("async timeout: {outcome}");


In [ ]:
// 10.4 Desk teardown: the async fleet is done with the stub. (`fs` persists from §2.1.)
fs::remove_dir_all("scratch23").unwrap();
println!("scratch23 removed: {}", !std::path::Path::new("scratch23").exists());


## 11. clap: Typed Command-Line Interfaces

Everything until now parsed argv by hand. **clap** is the ecosystem's `argparse`: you describe the interface once — *derive a struct* — and clap produces parsing, validation, defaults, `--help`, and error messages. The `derive` feature turns each field into an argument: `#[arg(long)]` for `--name`, doc comments become help text, types become validation.

### The interface as a type

`Ps` below is the `ps` subcommand's surface: `--format` constrained to enum variants via `ValueEnum`, `--last` a `u32` with a default. The **explicit type annotation on `ps`** is not style: under this kernel, evcxr's persistence analysis cannot infer the type of a `let` bound to a derive-generated constructor and refuses the cell — annotate, and the cell persists (§12 collects the receipt).


In [ ]:
// 11.1 Derive the interface: Parser for the struct, ValueEnum for the choices.
use clap::{Parser, ValueEnum};

/// List the stub fleet, docker-ps style.
#[derive(Parser, Debug)]
#[command(name = "stub-ps", about = "list the stub containers")]
struct Ps {
    /// output format: table or json lines
    #[arg(short, long, value_enum, default_value = "table")]
    format: Fmt,
    /// limit the listing to the first N containers
    #[arg(long, default_value = "12")]
    last: u32,
}

#[derive(ValueEnum, Clone, Debug)]
enum Fmt {
    Table,
    Json,
}

let ps: Ps = Ps::parse_from(["stub-ps", "--format", "json", "--last", "5"]);
println!("parsed: format={:?} last={}", ps.format, ps.last);

// defaults apply when flags are absent:
let defaults: Ps = Ps::parse_from(["stub-ps"]);
println!("defaults: format={:?} last={}", defaults.format, defaults.last);


### Errors as values: `try_parse_from`

`parse_from` **exits the process** on bad input — correct for a real `main`, fatal for a notebook kernel. `try_parse_from` is the library-shaped twin: the same validation, returned as `Result`. The error's `Display` is the exact text a user would see, suggestion list included. (The results are consumed inside a block — §12 explains why letting anything clap produced reach top level is kernel-hostile.)


In [ ]:
// 11.2 try_parse_from: clap's rejection, without the process exit.
// (Everything stays in one block: nothing produced from clap becomes a
// top-level variable -- derive-typed Results and even the rejected-value
// String both tripped evcxr's persistence analysis late in a session --
// §12's receipt. Print inside; bind nothing outside.)
{
    let bad = Ps::try_parse_from(["stub-ps", "--format", "yaml"]);
    match &bad {
        Ok(_) => println!("unexpectedly parsed"),
        Err(e) => println!("rejected: {}", e.to_string().lines().next().unwrap_or("")),
    }
    // full rejection text (the receipt):
    //   error: invalid value 'yaml' for '--format <FORMAT>'
    //     [possible values: table, json]
    let good = Ps::try_parse_from(["stub-ps", "--last", "3"]);
    println!("good parse last={}", good.as_ref().map(|p| p.last).unwrap_or(0));
}


### `--help` for free

The help text is generated from the struct: doc comments, defaults, value choices. `CommandFactory::command()` rebuilds clap's own view of the interface so it can be inspected (or `render_help`'d) without spawning anything. The help string stays inside a block, like every other clap product (§12).


In [ ]:
// 11.3 render_help: the generated --help, inspected as a String.
// Same block rule as 11.2: clap's output never becomes a top-level variable.
use clap::CommandFactory;

{
    let help = Ps::command().render_help().to_string();
    println!("mentions --format: {}", help.contains("--format"));
    println!("mentions --last: {}", help.contains("--last"));
    println!("first line: {}", help.lines().next().unwrap_or(""));
}


## 12. Reading the Compiler: E-Codes and Traps

| Symptom | Cause and fix |
|---|---|
| `cannot find type 'Command' in this scope` | `Command` is not in the prelude — `use std::process::Command;` in the first cell that needs it (§0.2). |
| `cannot find module or crate 'tokio'` | The `:dep tokio` cell has not run in this kernel session — crates are per-session; restart and go from the top. |
| cell refused: *"The variable `text` contains a reference with a non-static lifetime so can't be persisted"* | A top-level `let` holds a `Cow<str>` (from `from_utf8_lossy`) or any borrowed value. Add `.into_owned()` / `.to_string()`, or scope the binding inside a block (§5.2). |
| cell refused: *"Couldn't automatically determine type of variable `ps`"* | The `let` binds a derive-generated type (`Ps::parse_from(..)`). Annotate it: `let ps: Ps = ...` (§11.1). |
| cell wedged, or *"Subprocess terminated"* after an async cell | A `println!` inside `block_on` races the kernel's output capture. Compute inside, print after (§10; receipts in §13). |
| a later async cell hangs although each worked standalone | A second top-level `Runtime` while an earlier one persists: the session keeps every top-level binding, and once a runtime has waited on a child, the next one wedges. Scope every runtime — `let out = { let rt = Runtime::new().unwrap(); rt.block_on(..) };` — so each drops before the cell returns (§10). |
| `kill()` "failed" but the child is gone | It did not: kill-after-exit is `Ok(())` (§8.2). The child was already reaped by `wait()`. |
| bare program name is `NotFound` although the file is right there | Windows does not search the current directory for bare names — spell `./name.exe` (§8.3). |
| `error[E0425]: cannot find function 'version' in crate 'clap'` | `clap::version()` is not on this API surface — `CommandFactory::command()` is the introspection route (§11.3). |
| *"borrowed data escapes outside of function"* on a clap cell late in the session | A top-level binding produced from clap's types (a derive-typed `Result`, even an owned help `String`) — evcxr's persistence analysis chokes once the session is long. Keep every clap product in block scope: print inside the block, bind nothing outside (§11.2, §11.3). |


## 13. Mental Model: The Receipts

Every behavioural claim in this chapter was probed on rustc 1.98.1 through this exact kernel before being taught. The receipts:

| Claim | Receipt |
|---|---|
| kernel stdin blocks forever | probe cell with `stdin().read_line` timed out at 180 s: no bytes, no EOF |
| `\r\n` in captured pipes | `Output.stdout` of `cmd /C echo` ends `\r\n`; `str::lines()` strips it |
| missing program | `Err`, `kind = NotFound`, `message = "program not found"`; `unwrap` panics with the error's `Debug` |
| exit codes | `Some(0)`/`Some(2)` on the stub; **killed child on Windows = `Some(1)`** (Unix: `None`) |
| kill-after-exit | `Ok(())`, idempotent |
| piped stdin | write, drop, `wait_with_output` — `findstr` filtered exactly the AAPL line |
| `.env` reach | `STUB_VERSION` overridden for one child; parent's env untouched |
| lookup | `./name.exe` resolves; bare `name.exe` is `NotFound` in the same directory |
| desk build | `rustc --edition 2024 -O` on the stub: ~0.7 s first build, **second build the same** (raw rustc has no incremental cache) |
| pipe endings | the Rust stub writes bare `
`; `cmd /C echo` writes `
` — `str::lines()` strips both |
| async output/stream | `.output().await` and `next_line().await` behave as sync twins, print-after shape; runtimes scoped per cell — a second top-level `Runtime` while an earlier one persisted wedged the session (§12) |
| timeout + kill_on_drop | 400 ms timeout on `sleep 30`: miss → killed, `success = false` |
| clap derive | parse/defaults/`try_parse_from`/`render_help` verified in-kernel; error text `invalid value 'yaml' for '--format <FORMAT>'` |
| print-inside-`block_on` | wedged one probe run and tore down the kernel subprocess on a second — the shape is banned in this chapter |
| `Cow<str>` persistence | cell refused by evcxr until `.into_owned()` — the §5.2 block shape is the alternative |
| derive-type persistence | cell refused until `let ps: Ps` — annotation required under evcxr |


## 14. Cheat Sheet

```rust
use std::process::{Command, Stdio};

// run to completion, capture everything
let out = Command::new("./tool.exe").args(["ps", "--format", "{{json .}}"]).output()?;
let ok = out.status.success();                       // exit code as data
let text = String::from_utf8_lossy(&out.stdout).into_owned();  // bytes -> String

// streams: inherited (default) | piped() | null()
let mut child = Command::new("./tool.exe").arg("logs")
    .stdout(Stdio::piped()).spawn()?;                // start now
let stdout = child.stdout.take().unwrap();           // parent's pipe end
for line in std::io::BufReader::new(stdout).lines() { /* stream */ }
let st = child.wait()?;                              // reap

// feed a child's stdin
let mut c = Command::new("cmd").args(["/C", "findstr", "AAPL"])
    .stdin(Stdio::piped()).stdout(Stdio::piped()).spawn()?;
{ c.stdin.take().unwrap().write_all(b"data\n")?; }   // drop = EOF
let out = c.wait_with_output()?;

// child-specific world
Command::new("t").env("K", "v").env_remove("SECRET").current_dir("/tmp");

// lifecycle
child.try_wait()?;   // Ok(None) while running
child.kill()?;       // request; wait() still required

// async twins (tokio::process; tokio "full" carries it)
let out = tokio::process::Command::new("./t").output().await?;
tokio::time::timeout(d, child.wait()).await          // + kill_on_drop(true)

// clap (features "derive")
#[derive(Parser)] struct A { #[arg(long)] name: String }
let a: A = A::parse_from(["prog", "--name", "x"]);   // annotate under evcxr
A::try_parse_from(..)                                // errors as Result
```

| Python | Rust |
|---|---|
| `input(prompt)` | `stdin().read_line` in a binary (kernel: dead — §0) |
| `subprocess.run([..], capture_output=True)` | `Command::new(..).output()` |
| `p.returncode` | `out.status.code()` / `.success()` |
| `subprocess.Popen(..)` | `Command::new(..).spawn()` → `Child` |
| `p.communicate(input=b"..")` | `.take()`, write, drop, `wait_with_output()` |
| `env=` / `cwd=` | `.env(..)` / `.current_dir(..)` |
| `text=True` | `String::from_utf8_lossy(&out.stdout)` |
| `argparse` | `clap` derive |


## 15. Review

1. Why does `stdin().read_line()` compile in a notebook cell but never return? Which §0 receipt settles it, and where does the pattern work instead?
2. `.output()` returned `Err(ErrorKind::NotFound)`. Did the child fail, or did nothing launch? What in the type system forces you to answer before touching `stdout`?
3. A teammate's script kills a child and reads `code() == Some(1)` on Windows, then the same logic reports `None` on a Linux box. Explain both observations with one concept.
4. Why must the writing half of a piped stdin be dropped before `wait_with_output()`? What is the deadlock if not?
5. `String::from_utf8_lossy` returns `Cow<str>` — name the two situations it covers, and the kernel rule that makes `.into_owned()` mandatory in this notebook.
6. Bare `docker-stub.exe` fails with `NotFound` while `./docker-stub.exe` works from the same directory. What is the platform rule, and what attack does it defuse?
7. `kill_on_drop(true)` — what does the drop of a `tokio::process::Child` do with and without it, and which §8.2 rule is it the async answer to?
8. Why does `Ps::parse_from` need an explicit type annotation under evcxr when plain inference works in a compiled binary? What does that tell you about what the kernel does *between* cells?
9. `try_parse_from` vs `parse_from`: which is correct inside a library function, and why is the other one kernel-hostile?
10. The chapter's thesis: name three things Python's `subprocess` keyword arguments hide that Rust makes you hold as values.


### What Comes Next

Chapter 24 turns the **bytes** this chapter captured into structure: `regex` over text, CSV into rows, JSON into `HashMap`s and `Vec`s — the `docker ps --format "{{json .}}"` lines of §5 and §10 are exactly the feed it parses. Chapter 25 makes the discipline testable: the stub's contract, this chapter's receipts, become `#[test]`s. Chapter 26 closes the series with macros — the machinery `clap`'s `#[derive(Parser)]` was quietly using all along.

Carry forward: `Command` is a builder; `Output`/`ExitStatus` are data; `Stdio` picks the wiring; `Child` is a resource you own and must reap; stdout is bytes until decoded; print after `block_on`; and clap turns an interface into a type.
